# B2-flip — prompt từ kho donor thống nhất

Dùng bộ B2 tham chiếu đã truy hồi/fit whitening trên `research_freeze_20261101`, giữ nguyên target/graph/code demo và đảo nhãn **demo mới**. Không gọi LLM, Joern hoặc encoder CodeT5.

Xuất riêng `output/bundles_frozen_donor_20261101/grace_b2_flip_frozen_donor_inputs.zip`; bundle cũ được giữ để đối chiếu checkpoint I-06.


In [ ]:
from pathlib import Path
import sys, json, subprocess, importlib
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.1', 'sentencepiece'], check=True)
PROJECT_ROOT_OVERRIDE = None
candidates = [Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p/'04_Arm_B2_Flip/prepare_frozen_flip.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook trong repo hoặc đặt PROJECT_ROOT_OVERRIDE trỏ GRACE-summary')
sys.path.insert(0, str(ROOT/'04_Arm_B2_Flip'))
import prepare_frozen_flip
importlib.reload(prepare_frozen_flip)
REFERENCE_ROOT = prepare_frozen_flip.REFERENCE_ROOT
FROZEN_ROOT = prepare_frozen_flip.FROZEN_ROOT
LEGACY_ROOT = prepare_frozen_flip.LEGACY_ROOT
OUT_ROOT = prepare_frozen_flip.OUT_ROOT
LOCAL_FILES_ONLY = True  # Tokenizer đã cache; đặt False nếu cần tải đúng revision đã pin.
print('B2 của donor mới:', REFERENCE_ROOT)
print('Output mới:', OUT_ROOT)


## Kiểm tra phiên donor và bộ B2 mới

Xác minh hash B2 tham chiếu, cache donor và snapshot đã kiểm toán. Bộ tham chiếu đã tái dùng cache Joern/CodeT5 và fit lại whitening trên kho mới. Notebook này chỉ dựng biến thể flip và đếm token bằng đúng tokenizer; không thực hiện lại retrieval.


In [ ]:
snapshot_sha, proofs = prepare_frozen_flip.inspect_sources(REFERENCE_ROOT, FROZEN_ROOT)
print('Donor snapshot SHA-256:', snapshot_sha)
for dataset, proof in proofs.items():
    print(dataset, 'donor mới:', len(proof['labels']))


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(prepare_frozen_flip.TOKENIZER,
    revision=prepare_frozen_flip.TOKENIZER_REVISION, local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False)
assert tokenizer.chat_template
reference = next(prepare_frozen_flip.rows(REFERENCE_ROOT/'devign/validation/prompts.jsonl'))
demo_id = reference['demo_ids'][0]
preview = prepare_frozen_flip.swap_demo_label(reference['prompt'], proofs['devign']['labels'][demo_id])
print('Demo mới:', demo_id)
print(preview)


In [ ]:
importlib.reload(prepare_frozen_flip)
bundles, archive = prepare_frozen_flip.prepare(tokenizer, REFERENCE_ROOT, FROZEN_ROOT, LEGACY_ROOT, OUT_ROOT)
print('Upload archive này lên Kaggle:', archive)
for path in bundles:
    print(path)
print((OUT_ROOT/'prepare_summary.json').read_text(encoding='utf-8'))


## Đầu ra

Đủ 10.011 prompt trên bốn split. So với phiên cũ: Devign validation đổi 28, Devign test đổi 38, REVEAL không đổi. Mỗi dòng lưu `previous_prompt_sha256` và `previous_demo_ids` để notebook inference xác minh checkpoint cũ trước khi quyết định giữ/chạy lại; hash lạ phải bị từ chối.

Upload ZIP **frozen_donor** mới, checkpoint `grace_b2_flip_checkpoint.zip` của I-06 và model GGUF vào Input của notebook 02. Notebook 02 mặc định bật resume, chuyển checkpoint sang thư mục phiên mới, ưu tiên 33 mẫu đã chạy nhưng đổi prompt rồi tiếp tục phần thiếu. Giữ nguyên bundle/output lịch sử.
